In [ ]:
import math
from dataclasses import dataclass

@dataclass
class HW:
    peak_ops_per_cycle: float   # 阵列每周期能完成的有效 ops（MAC*2 或 bit-op 自定义）
    ext_bw_BpC: float           # 外部带宽 Bytes/cycle
    icw_BpC: float              # Input 注入带宽 Bytes/cycle（片上->阵列）
    wuw_BpC: float              # Weight 更新带宽 Bytes/cycle（片外/片上->CIM）
    IS_bytes: int               # Input SRAM 容量
    OS_bytes: int               # Output SRAM 容量
    SCR: int                    # storage-compute ratio（抽象参数）
    bytes_x: int = 2            # X 每元素字节数（FP16=2）
    bytes_w: int = 2            # W 每元素字节数
    bytes_y: int = 2            # Y/psum 每元素字节数

@dataclass
class Gemm:
    M: int
    N: int
    K: int

def ceildiv(a, b): 
    return (a + b - 1) // b

def simulate_gemm(hw: HW, g: Gemm, mt=128, nt=128, kt=256, policy="AF"):
    """
    policy: "AF" or "PF"
    AF: 优先让同一输出tile在片上累加，若OS不够则spill
    PF: 优先输入复用（简化为更容易触发并行psum压力，等效更大OS需求）
    """
    # tile sizes sanity with IS/OS constraints (simple clamp)
    # IS needs to hold X tile (mt*kt)
    while mt * kt * hw.bytes_x > hw.IS_bytes and mt > 1:
        mt //= 2
    # OS needs to hold output tile (mt*nt) psum
    # PF 让并行 psum 压力更大：等效放大一个系数（很粗，但能反映趋势）
    pf_factor = hw.SCR if policy == "PF" else 1
    while mt * nt * hw.bytes_y * pf_factor > hw.OS_bytes and nt > 1:
        nt //= 2

    n_mt = ceildiv(g.M, mt)
    n_nt = ceildiv(g.N, nt)
    n_kt = ceildiv(g.K, kt)

    total_cycles = 0
    stall_in = 0
    stall_wu = 0
    stall_out = 0
    compute_cycles = 0

    # loop tiles
    for i in range(n_mt):
        cur_m = mt if i < n_mt - 1 else (g.M - i * mt)
        for j in range(n_nt):
            cur_n = nt if j < n_nt - 1 else (g.N - j * nt)

            # output tile size
            out_bytes = cur_m * cur_n * hw.bytes_y

            # determine if psum fits
            os_need = out_bytes * (hw.SCR if policy == "PF" else 1)
            psum_fits = os_need <= hw.OS_bytes

            # k-tiles accumulation
            for kk in range(n_kt):
                cur_k = kt if kk < n_kt - 1 else (g.K - kk * kt)

                # bytes to bring X and W for this k-slice
                x_bytes = cur_m * cur_k * hw.bytes_x
                w_bytes = cur_k * cur_n * hw.bytes_w

                # input fetch from off-chip (if not resident): simplified assume every k-slice fetch X
                # You can later improve with reuse across j.
                c_ext_x = math.ceil(x_bytes / hw.ext_bw_BpC)
                # then inject to array limited by icw
                c_inject = math.ceil(x_bytes / hw.icw_BpC)
                c_in = max(c_ext_x, c_inject)
                stall_in += c_in

                # weight update/copy to CIM limited by WUW; SCR increases "stored choices" but compute uses 1.
                # Here we model that higher SCR makes you potentially update/select more often:
                wu_mult = 1 if policy == "AF" else hw.SCR  # PF tends to exploit SCR more aggressively
                c_wu = math.ceil((w_bytes * wu_mult) / hw.wuw_BpC)
                stall_wu += c_wu

                # compute ops for this slice: 2*M*N*Kslice (MAC=mul+add)
                ops = 2 * cur_m * cur_n * cur_k
                c_comp = math.ceil(ops / hw.peak_ops_per_cycle)
                compute_cycles += c_comp

                # output spill/reload if OS cannot hold psum
                if not psum_fits:
                    # spill psum out each slice, reload next slice (except last)
                    wr = math.ceil(out_bytes / hw.ext_bw_BpC)
                    stall_out += wr  # write
                    if kk != n_kt - 1:
                        rd = math.ceil(out_bytes / hw.ext_bw_BpC)
                        stall_out += rd  # read back

    total_cycles = stall_in + stall_wu + compute_cycles + stall_out

    return {
        "mt": mt, "nt": nt, "kt": kt,
        "policy": policy,
        "total_cycles": total_cycles,
        "compute_cycles": compute_cycles,
        "stall_in": stall_in,
        "stall_wu": stall_wu,
        "stall_out": stall_out,
        "utilization": compute_cycles / total_cycles if total_cycles > 0 else 0.0
    }

# Example sweep idea:
# for IS in [64<<10, 128<<10, 256<<10]:  # 64KB, 128KB, 256KB
#   for OS in [64<<10, 128<<10, 256<<10]:
#     for SCR in [1,2,4,8]:
#       hw = HW(peak_ops_per_cycle=1e6, ext_bw_BpC=256, icw_BpC=512, wuw_BpC=256, IS_bytes=IS, OS_bytes=OS, SCR=SCR)
#       print(simulate_gemm(hw, Gemm(256,1024,1024), policy="AF"))

In [ ]:
import torch
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any

@dataclass
class StreamResamplerState:
    B: int
    L: int
    out_len: int
    left: torch.Tensor      # [out_len] int64
    right: torch.Tensor     # [out_len] int64
    w: torch.Tensor         # [out_len] float32
    buf: dict               # idx -> tensor [B]
    i_in: int               # next input index expected
    j_out: int              # next output index to produce
    cycles: int             # total cycles elapsed

class LinResample1DStreamer:
    def __init__(self, L: int, out_len: int, B: int = 1, device=None, mode: str = "linspace"):
        assert mode in ("linspace", "iter")
        self.device = device if device is not None else torch.device("cpu")
        self.mode = mode
        self.reset(L=L, out_len=out_len, B=B)

    def reset(self, L: int, out_len: int, B: int = 1):
        self.L = int(L)
        self.out_len = int(out_len)
        self.B = int(B)

        # src_pos generation
        if self.out_len == 1:
            src_pos = torch.zeros((1,), dtype=torch.float32, device=self.device)
        else:
            if self.mode == "linspace":
                src_pos = torch.linspace(0, self.L - 1, steps=self.out_len,
                                         dtype=torch.float32, device=self.device)
            else:
                # iterative float32 accumulation
                step = torch.tensor((self.L - 1) / (self.out_len - 1),
                                    dtype=torch.float32, device=self.device)
                src_pos = torch.empty((self.out_len,), dtype=torch.float32, device=self.device)
                pos = torch.tensor(0.0, dtype=torch.float32, device=self.device)
                for j in range(self.out_len):
                    src_pos[j] = pos
                    pos = pos + step
                src_pos[-1] = torch.tensor(float(self.L - 1), dtype=torch.float32, device=self.device)

        left = torch.floor(src_pos).to(torch.int64).clamp(0, self.L - 1)
        right = (left + 1).clamp(0, self.L - 1)
        w = (src_pos - left.to(torch.float32)).to(torch.float32)

        self.state = StreamResamplerState(
            B=self.B, L=self.L, out_len=self.out_len,
            left=left, right=right, w=w,
            buf={}, i_in=0, j_out=0, cycles=0
        )

    @property
    def done(self) -> bool:
        return self.state.j_out >= self.state.out_len

    def step(self, x_i: Optional[torch.Tensor] = None) -> Tuple[bool, Optional[torch.Tensor], Dict[str, Any]]:
        st = self.state

        # snapshot current output requirement
        pre_j = st.j_out
        need_l = int(st.left[pre_j].item()) if pre_j < st.out_len else None
        need_r = int(st.right[pre_j].item()) if pre_j < st.out_len else None
        need_w = float(st.w[pre_j].item()) if pre_j < st.out_len else None

        # push one input sample
        pushed = False
        pushed_idx = None
        if x_i is not None:
            assert x_i.shape == (st.B,), f"x_i must be shape [B]={st.B}"
            assert x_i.dtype == torch.float32, "x_i must be FP32"
            assert st.i_in < st.L, "All inputs already pushed"
            pushed = True
            pushed_idx = st.i_in
            st.buf[st.i_in] = x_i.to(self.device).clone()
            st.i_in += 1

        # check availability after push
        have_l = (need_l in st.buf) if need_l is not None else False
        have_r = (need_r in st.buf) if need_r is not None else False

        # try produce one output
        y = None
        valid = False
        produced_j = None
        if pre_j < st.out_len and have_l and have_r:
            wl = st.w[pre_j]
            y = (1.0 - wl) * st.buf[need_l] + wl * st.buf[need_r]
            valid = True
            produced_j = pre_j
            st.j_out += 1

            # optional GC for old samples (monotonic mapping)
            min_keep = max(0, need_l - 2)
            for k in list(st.buf.keys()):
                if k < min_keep:
                    del st.buf[k]

        st.cycles += 1

        info = {
            "cycle": st.cycles,
            "pushed": pushed,
            "pushed_idx": pushed_idx,
            "pre_j_out": pre_j,
            "produced": valid,
            "produced_j": produced_j,
            "need_left": need_l,
            "need_right": need_r,
            "need_w": need_w,
            "have_left": have_l,
            "have_right": have_r,
            "buf_min": min(st.buf.keys()) if st.buf else None,
            "buf_max": max(st.buf.keys()) if st.buf else None,
            "buf_size": len(st.buf),
            "i_in_next": st.i_in,
            "j_out_next": st.j_out,
        }
        return valid, y, info

def lin_resample_1d_ref(x: torch.Tensor, out_len: int) -> torch.Tensor:
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape
    if L == out_len:
        return x
    src_pos = torch.linspace(0, L - 1, steps=out_len, device=x.device, dtype=x.dtype)
    left = torch.floor(src_pos).long().clamp(0, L - 1)
    right = (left + 1).clamp(0, L - 1)
    w = (src_pos - left.to(x.dtype))[None, :]
    return (1 - w) * x[:, left] + w * x[:, right]

def run_streaming_full_verbose(
    x: torch.Tensor,
    out_len: int,
    mode: str = "linspace",
    max_print_cycles: int = 80,
    print_values: bool = False,
) -> Tuple[torch.Tensor, List[Dict[str, Any]]]:
    """
    Feed one input per cycle; collect outputs.
    Also returns per-cycle logs. Prints first max_print_cycles cycles.
    """
    assert x.dtype == torch.float32
    if x.ndim == 1:
        x = x[None, :]
    B, L = x.shape

    streamer = LinResample1DStreamer(L=L, out_len=out_len, B=B, device=x.device, mode=mode)

    ys = []
    logs = []
    i = 0
    while not streamer.done:
        x_i = x[:, i] if i < L else None
        if i < L:
            i += 1

        valid, y, info = streamer.step(x_i)
        if valid:
            ys.append(y)
            if print_values:
                info["y0"] = float(y[0].item()) if B == 1 else None

        logs.append(info)

        c = info["cycle"]
        if c <= max_print_cycles:
            if info["produced"]:
                msg = (f"[C{c:04d}] push={info['pushed']} idx={info['pushed_idx']} | "
                       f"OUT j={info['produced_j']} uses (l={info['need_left']}, r={info['need_right']}, w={info['need_w']:.6f}) | "
                       f"buf=[{info['buf_min']},{info['buf_max']}] size={info['buf_size']}")
                if print_values and "y0" in info:
                    msg += f" | y[0]={info['y0']:.6f}"
            else:
                msg = (f"[C{c:04d}] push={info['pushed']} idx={info['pushed_idx']} | "
                       f"STALL need (l={info['need_left']}, r={info['need_right']}, w={info['need_w']:.6f}) "
                       f"have(l={info['have_left']}, r={info['have_right']}) | "
                       f"buf=[{info['buf_min']},{info['buf_max']}] size={info['buf_size']}")
            print(msg)

    y_out = torch.stack(ys, dim=1)  # [B,out_len]
    return y_out, logs